In [ ]:
# 1. Montar Drive & configurar caminhos
from google.colab import drive
import os, subprocess
from datetime import datetime

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else (
    "MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive"
)
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
RAW_PATH = os.path.join(BASE_PATH, "data_raw")
os.makedirs(PROC_PATH, exist_ok=True)

NB_NAME = "18_backtest_simulation"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print("PROC_PATH:", PROC_PATH)

In [ ]:
# 2. Bibliotecas e parâmetros
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

paths = {
    "labels": os.path.join(PROC_PATH, "labels_y_daily.csv"),
    "oof": os.path.join(PROC_PATH, "16_oof_predictions.csv"),
}
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {path}")

with open(os.path.join(PROC_PATH, "results_16_models_tfidf.json"), "r", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

STRATEGIES = [
    {"name": "long_only_55", "long_th": 0.55, "short_th": 0.45, "allow_short": False, "cost": 0.0005},
    {"name": "long_only_60", "long_th": 0.60, "short_th": 0.40, "allow_short": False, "cost": 0.0005},
    {"name": "long_short_sym", "long_th": 0.55, "short_th": 0.45, "allow_short": True, "cost": 0.0007},
]

print(json.dumps(STRATEGIES, indent=2))

In [ ]:
# 3. Carregar dados e preparar base
labels = pd.read_csv(paths["labels"], parse_dates=["day"])
oof = pd.read_csv(paths["oof"], parse_dates=["day"])

df = oof.merge(
    labels.drop(columns=["y"], errors="ignore"),
    on="day",
    how="left",
)
if df.empty:
    raise ValueError("Sem previsões para o backtest.")

if "ret_next" not in df.columns:
    raise ValueError("labels_y_daily.csv precisa ter a coluna ret_next.")

df.sort_values(["model", "day"], inplace=True)
df["sentiment"] = df["proba"] * 2 - 1

display(df.head())
print(df.groupby("model")["day"].nunique())

In [ ]:
# 4. Funções utilitárias
from typing import Dict


def compute_metrics(returns: pd.Series, equity_curve: pd.Series) -> Dict[str, float]:
    if returns.empty:
        return {"cagr": np.nan, "vol": np.nan, "sharpe": np.nan, "max_dd": np.nan, "hit_ratio": np.nan}
    total_return = (1 + returns).prod() - 1
    years = len(returns) / 252
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else np.nan
    vol = returns.std(ddof=0) * np.sqrt(252)
    avg = returns.mean() * 252
    sharpe = avg / vol if vol and vol > 0 else np.nan
    hit_ratio = (returns > 0).mean()
    running_max = equity_curve.cummax()
    dd = equity_curve / running_max - 1
    max_dd = dd.min()
    return {
        "total_return": total_return,
        "cagr": cagr,
        "vol": vol,
        "sharpe": sharpe,
        "max_dd": max_dd,
        "hit_ratio": hit_ratio,
    }


def run_strategy(data: pd.DataFrame, config: Dict[str, float]) -> pd.DataFrame:
    df_strat = data.copy().reset_index(drop=True)
    long_th = config["long_th"]
    short_th = config["short_th"]
    allow_short = config.get("allow_short", True)
    cost = config.get("cost", 0.0005)

    raw_signal = np.where(
        df_strat["proba"] >= long_th,
        1,
        np.where(df_strat["proba"] <= short_th, -1 if allow_short else 0, 0),
    )

    df_strat["signal"] = raw_signal
    df_strat["turnover"] = np.abs(np.diff(np.insert(raw_signal, 0, 0)))
    df_strat["cost"] = df_strat["turnover"] * cost
    df_strat["strategy_ret"] = df_strat["signal"] * df_strat["ret_next"] - df_strat["cost"]
    df_strat["equity"] = (1 + df_strat["strategy_ret"]).cumprod()

    return df_strat

In [ ]:
# 5. Rodar simulações para cada modelo x estratégia
perf_rows = []
equity_records = []
daily_records = []

for model_name, sub in df.groupby("model"):
    ordered = sub.sort_values("day").reset_index(drop=True)
    for config in STRATEGIES:
        strat_name = config["name"]
        result = run_strategy(ordered, config)
        metrics = compute_metrics(result["strategy_ret"], result["equity"])
        perf_rows.append({
            "model": model_name,
            "strategy": strat_name,
            "n_days": len(result),
            **metrics,
        })

        equity_records.append({
            "model": model_name,
            "strategy": strat_name,
            "day": result["day"],
            "equity": result["equity"],
        })
        tmp = result.copy()
        tmp["strategy"] = strat_name
        tmp["model"] = model_name
        daily_records.append(tmp)

perf_df = pd.DataFrame(perf_rows)
display(perf_df.sort_values(["model", "strategy"]))

In [ ]:
# 6. Gráfico de curvas de capital
if equity_records:
    traces = []
    for rec in equity_records:
        traces.append(
            go.Scatter(
                x=rec["day"],
                y=rec["equity"],
                mode="lines",
                name=f"{rec['model']} | {rec['strategy']}",
            )
        )
    equity_fig = go.Figure(data=traces)
    equity_fig.update_layout(title="Curvas de capital - Estratégias", yaxis_title="Equity (R$ 1 inicial)")
    equity_fig.show()
else:
    equity_fig = None

In [ ]:
# 7. Salvar resultados
perf_file = os.path.join(PROC_PATH, "18_backtest_results.csv")
daily_file = os.path.join(PROC_PATH, "18_backtest_daily_curves.csv")
fig_file = os.path.join(PROC_PATH, "18_backtest_equity.html")

perf_df.to_csv(perf_file, index=False, encoding="utf-8")
if daily_records:
    daily_df = pd.concat(daily_records, ignore_index=True)
    daily_df.to_csv(daily_file, index=False, encoding="utf-8")
else:
    daily_df = pd.DataFrame()

if equity_fig is not None:
    equity_fig.write_html(fig_file)

print("Arquivos salvos:")
print(perf_file)
print(daily_file)
print(fig_file)

In [ ]:
# 8. Resumo final
from IPython.display import Markdown

best = perf_df.sort_values("sharpe", ascending=False).head(3)
best_md = best.to_markdown(index=False)

md = "**18_backtest_simulation concluído ✅**
"
md += "- Estratégias testadas: " + ", ".join([c["name"] for c in STRATEGIES]) + "
"
md += "- Best Sharpe (Top 3):
" + best_md + "
"
md += "
**Próximo:** `20_final_dashboard_analysis` → consolidar métricas e gerar painel final."

Markdown(md)